In [ ]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import scipy.io as sio
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [ ]:
def load_ttbi_dataset(base_dir='data', dataset_name='data_noise', 
                      damage_files=range(0, 61), sensor_types=['bogie'], 
                      dofs=[0], n_passages=200):
    """
    Loads TTBI 2D dataset files and formats them for a multi-channel 1D CNN.
    
    Args:
        base_dir (str): The root directory containing the datasets.
        dataset_name (str): The specific folder to load (e.g., 'data_noise_speed').
        damage_files (list or range): The file numbers to load (1 to 61).
        sensor_type (str): 'bogie' (AceleracaoPrimVag) or 'wheel' (AcelRodaPrimVag).
        dofs (list): List of DOF indices to extract (e.g., [0], [0, 1], [0, 1, 2]).
        n_passages (int): Number of passages to extract per file.
        
    Returns:
        X (np.ndarray): The features array of shape (Samples, Channels, Sequence_Length).
        y (np.ndarray): The labels array of shape (Samples,) representing damage %.
    """
    
    dataset_path = os.path.join(base_dir, dataset_name)
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset folder not found: {dataset_path}")

    X_list = []
    y_list = []

    # print(f"Loading '{dataset_name}' | Sensors: {sensor_types} | DOFs: {dofs}...")

    for damage_label in damage_files:
        # Format the file name, e.g., 0001.mat
        filename = f"{damage_label+1:04d}.mat"
        filepath = os.path.join(dataset_path, filename)
        
        try:
            mat = sio.loadmat(filepath)
            
            # Navigate the deeply nested MATLAB struct
            # mat['data'] is usually a 1x1 object array, containing the struct fields
            data_struct = mat['data'][0, 0]
            
            # Use the first sensor to determine the number of available passages
            first_field = 'AcelPrimVag' if sensor_types[0] == 'bogie' else 'AcelRodaPrimVag'
            available_passages = data_struct[first_field].shape[1]
            passages_to_load = min(n_passages, available_passages)
            
            for p in range(passages_to_load):
                combined_channels = []
                
                # Loop through the requested sensors and stack them
                for s_type in sensor_types:
                    field_name = 'AcelPrimVag' if s_type == 'bogie' else 'AcelRodaPrimVag'
                    sensor_data_cells = data_struct[field_name]
                    passage_matrix = sensor_data_cells[0, p]
                    
                    # Extract requested DOFs
                    selected_dofs = passage_matrix[dofs, :]
                    combined_channels.append(selected_dofs)
                
                # Stack all channels vertically (e.g., Bogie DOF 1 + Wheel DOF 1 = 2 Channels)
                X_list.append(np.vstack(combined_channels))
                y_list.append(damage_label)
                
        except FileNotFoundError:
            print(f"  [!] Missing file: {filename}")
        except KeyError:
            print(f"  [!] Field '{field_name}' not found in {filename}")
        except Exception as e:
            print(f"  [!] Error processing {filename}: {e}")

    # Convert to PyTorch-friendly NumPy arrays
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64) # int64 is standard for PyTorch classification labels
    
    # print(f"Successfully loaded {len(y)} samples. Shape: {X.shape}")
    
    return X, y

# --- Example Usage ---
if __name__ == '__main__':
    # Example: Load the healthy case (0), 10% damage (10), and 60% damage (60)
    # Extracting DOF 0 and DOF 2 from the Bogie across the noisy dataset
    test_cases = [0, 10, 60]
    
    X_data, y_labels = load_ttbi_dataset(
        base_dir='data',
        dataset_name='data_noise',
        damage_files=test_cases,
        sensor_types='bogie',
        dofs=[0, 2], 
        n_passages=200
    )

In [ ]:
import numpy as np
from scipy.interpolate import interp1d
import scipy.signal as signal
from sklearn.preprocessing import MinMaxScaler
import pickle

class TTBIPreprocessor:
    def __init__(self, method='raw', n_segments=512, cwt_scales=64):
        """
        Initializes the preprocessor for the TTBI ablation study.
        
        Args:
            method (str): 'raw', 'paa', 'fft', or 'cwt'.
            n_segments (int): The target length for PAA downsampling (or FFT bin count).
            cwt_scales (int): Number of frequency scales for the Wavelet Transform.
        """
        self.method = method.lower()
        self.n_segments = n_segments
        self.cwt_scales = cwt_scales
        self.scaler = MinMaxScaler(feature_range=(0, 1))
        self.is_fit = False

    def _apply_paa(self, X):
        """Downsamples the sequence length using Piecewise Aggregate Approximation (Interpolation)."""
        samples, channels, length = X.shape
        if length == self.n_segments:
            return X
            
        x_old = np.linspace(0, 1, length)
        x_new = np.linspace(0, 1, self.n_segments)
        
        # Fully vectorized interpolation along the last axis (axis=2)
        f_interp = interp1d(x_old, X, axis=2, kind='linear')
        X_paa = f_interp(x_new)
        return X_paa.astype(np.float32)

    def _apply_fft(self, X):
        """Computes the frequency magnitude spectrum using Fast Fourier Transform."""
        # rfft automatically computes only the positive frequencies for real inputs
        fft_coeffs = np.fft.rfft(X, axis=2)
        fft_mag = np.abs(fft_coeffs)
        
        # If the resulting bins don't match our target, interpolate them
        return self._apply_paa(fft_mag)

    def _apply_cwt(self, X):
        """
        Generates 2D Scalograms using the Continuous Wavelet Transform (Morlet wavelet).
        Note: Applies PAA first to prevent RAM Out-Of-Memory crashes.
        """
        print("  -> Downsampling spatial length before CWT to save memory...")
        X_downsampled = self._apply_paa(X)
        samples, channels, length = X_downsampled.shape
        
        widths = np.arange(1, self.cwt_scales + 1)
        
        # Output shape will be (Samples, Channels, Scales, Length)
        X_cwt = np.zeros((samples, channels, self.cwt_scales, length), dtype=np.float32)
        
        print(f"  -> Computing CWT Scalograms (Shape: {X_cwt.shape})...")
        for i in range(samples):
            for c in range(channels):
                # Using standard Morlet wavelet
                cwt_matrix = signal.cwt(X_downsampled[i, c, :], signal.morlet2, widths)
                X_cwt[i, c, :, :] = np.abs(cwt_matrix)
                
        return X_cwt

    def transform(self, X, fit_scaler=False):
        """
        Applies the selected signal processing method and scales the data.
        
        Args:
            X (np.ndarray): Input data of shape (Samples, Channels, Sequence_Length)
            fit_scaler (bool): If True, fits the MinMaxScaler. If False, only transforms.
            
        Returns:
            X_scaled (np.ndarray): Processed and scaled data ready for PyTorch.
        """
        # print(f"Applying '{self.method.upper()}' preprocessing...")
        
        # 1. Signal Processing
        if self.method == 'raw':
            X_processed = X
        elif self.method == 'paa':
            X_processed = self._apply_paa(X)
        elif self.method == 'fft':
            X_processed = self._apply_fft(X)
        elif self.method == 'cwt':
            X_processed = self._apply_cwt(X)
        else:
            raise ValueError(f"Unknown preprocessing method: {self.method}")

        # 2. Scaling (MinMaxScaler expects 2D data)
        # We flatten the Channels/Sequences into a single feature vector per sample
        original_shape = X_processed.shape
        samples = original_shape[0]
        
        X_flat = X_processed.reshape(samples, -1)
        
        if fit_scaler:
            X_scaled_flat = self.scaler.fit_transform(X_flat)
            self.is_fit = True
        else:
            if not self.is_fit:
                raise RuntimeError("Scaler has not been fitted yet! Pass fit_scaler=True for training data.")
            X_scaled_flat = self.scaler.transform(X_flat)
            
        # Reshape back to the 3D (or 4D for CWT) tensor shape expected by PyTorch CNNs
        X_scaled = X_scaled_flat.reshape(original_shape)
        
        return X_scaled

    def save_scaler(self, filepath):
        """Saves the fitted scaler for the Digital Twin online phase."""
        with open(filepath, 'wb') as f:
            pickle.dump(self.scaler, f)
        print(f"Scaler saved to {filepath}")

In [ ]:
class Space2Vec(nn.Module):
    """
    Learnable spatial embedding layer (adapted from Time2Vec).
    """
    def __init__(self, seq_len, out_features=8):
        super(Space2Vec, self).__init__()
        self.seq_len = seq_len
        self.out_features = out_features
        
        # Learnable parameters for the linear term (1 feature)
        self.w_linear = nn.Parameter(torch.randn(1, 1))
        self.p_linear = nn.Parameter(torch.randn(1, 1))
        
        # Learnable parameters for the periodic terms (out_features - 1)
        self.w_periodic = nn.Parameter(torch.randn(out_features - 1, 1))
        self.p_periodic = nn.Parameter(torch.randn(out_features - 1, 1))

    def forward(self, x_space):
        """
        Args:
            x_space: A spatial index tensor of shape (Batch, 1, Sequence_Length)
                     representing normalized positions from 0.0 to 1.0.
        Returns:
            Space2Vec embeddings of shape (Batch, out_features, Sequence_Length)
        """
        # Linear feature: shape (Batch, 1, Sequence_Length)
        linear = self.w_linear * x_space + self.p_linear
        
        # Periodic features: shape (Batch, out_features - 1, Sequence_Length)
        periodic = torch.sin(self.w_periodic * x_space + self.p_periodic)
        
        # Concatenate along the channel dimension (dim=1)
        # Final shape: (Batch, out_features, Sequence_Length)
        return torch.cat([linear, periodic], dim=1)


class SpaceAware1DCNN(nn.Module):
    """
    Dynamic 1D CNN that optionally concatenates Space2Vec embeddings to the input.
    """
    def __init__(self, n_segments, n_classes, in_channels, params, use_space2vec=True, s2v_features=8):
        super(SpaceAware1DCNN, self).__init__()
        self.params = params
        self.use_space2vec = use_space2vec
        self.n_segments = n_segments
        
        self.layers = nn.ModuleList()
        current_seq_len = n_segments
        
        # 1. Setup Space2Vec
        if self.use_space2vec:
            self.space2vec = Space2Vec(seq_len=n_segments, out_features=s2v_features)
            # We add the new spatial features to the original sensor channels
            cnn_in_channels = in_channels + s2v_features
        else:
            cnn_in_channels = in_channels
            
        # 2. Build Dynamic Convolutional Layers
        n_conv_layers = self.params['n_conv_layers']
        for i in range(n_conv_layers):
            out_channels = self.params[f'n_filters_l{i}']
            kernel_size = self.params[f'kernel_size_l{i}']
            
            self.layers.append(nn.Conv1d(cnn_in_channels, out_channels, kernel_size=kernel_size, padding='same'))
            self.layers.append(nn.ReLU())
            
            if self.params.get(f'pooling_l{i}', False):
                self.layers.append(nn.MaxPool1d(kernel_size=2, stride=2))
                current_seq_len = current_seq_len // 2
                
            cnn_in_channels = out_channels
            
        # 3. Build Dynamic Dense Layers
        self.layers.append(nn.Flatten())
        flattened_size = current_seq_len * cnn_in_channels
        
        n_dense_layers = self.params['n_dense_layers']
        in_features = flattened_size
        
        for i in range(n_dense_layers):
            out_features = self.params[f'n_dense_units_l{i}']
            self.layers.append(nn.Linear(in_features, out_features))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(self.params.get(f'dropout_l{i}', 0.2)))
            in_features = out_features
            
        self.layers.append(nn.Linear(in_features, n_classes))
        
    def forward(self, x):
        """
        Args:
            x: Input sensor signals of shape (Batch, Channels, Sequence_Length)
        """
        if self.use_space2vec:
            batch_size = x.size(0)
            
            # Generate a normalized spatial vector [0.0, ..., 1.0] representing the bridge
            # Shape: (Sequence_Length)
            space_vector = torch.linspace(0, 1, steps=self.n_segments, device=x.device)
            
            # Broadcast to shape (Batch, 1, Sequence_Length) so it matches the input batch
            space_vector = space_vector.view(1, 1, -1).expand(batch_size, 1, -1)
            
            # Generate the embeddings: Shape (Batch, s2v_features, Sequence_Length)
            s2v_embeddings = self.space2vec(space_vector)
            
            # Concatenate sensor data and spatial embeddings along the channel dimension
            # Shape becomes (Batch, Channels + s2v_features, Sequence_Length)
            x = torch.cat([x, s2v_embeddings], dim=1)
            
        # Pass through the standard CNN layers
        for layer in self.layers:
            x = layer(x)
            
        return x

class Simple2DCNN(nn.Module):
    """
    Dynamic 2D CNN designed specifically for Continuous Wavelet Transform (CWT) scalograms.
    Input shape expects: (Batch_Size, Channels, Scales_Height, Sequence_Width)
    """
    def __init__(self, in_channels, n_classes, params, image_height=64, image_width=512):
        super(Simple2DCNN, self).__init__()
        self.params = params
        self.layers = nn.ModuleList()
        
        # Track the spatial dimensions to calculate the flatten size mathematically
        current_h = image_height
        current_w = image_width
        current_channels = in_channels
        
        n_conv_layers = self.params['n_conv_layers']
        
        # 1. Dynamic 2D Convolutional Layers
        for i in range(n_conv_layers):
            out_channels = self.params[f'n_filters_l{i}']
            # Optuna suggests a single integer (e.g., 3), which PyTorch interprets as a (3, 3) square kernel
            k_size = self.params[f'kernel_size_l{i}']
            
            # padding='same' ensures the convolution doesn't shrink the image dimensions
            self.layers.append(nn.Conv2d(current_channels, out_channels, kernel_size=k_size, padding='same'))
            self.layers.append(nn.ReLU())
            
            # 2x2 Max Pooling cuts the height and width exactly in half
            if self.params.get(f'pooling_l{i}', False):
                self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
                current_h = current_h // 2
                current_w = current_w // 2
                
            current_channels = out_channels
            
        # 2. Flatten for the Dense Layers
        self.layers.append(nn.Flatten())
        flattened_size = current_channels * current_h * current_w
        
        # 3. Dynamic Dense (Linear) Layers
        n_dense_layers = self.params['n_dense_layers']
        in_features = flattened_size
        
        for i in range(n_dense_layers):
            out_features = self.params[f'n_dense_units_l{i}']
            self.layers.append(nn.Linear(in_features, out_features))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(self.params.get(f'dropout_l{i}', 0.2)))
            in_features = out_features
            
        # 4. Final Classification Output
        self.layers.append(nn.Linear(in_features, n_classes))
        
    def forward(self, x):
        """
        Args:
            x: Input scalograms of shape (Batch, Channels, Height, Width)
        """
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
# # 1. Load multi-channel data (e.g., Vertical Accel and Pitch)
# X_raw, y = load_ttbi_dataset(dofs=[0, 2]) 

# # 2. Initialize Preprocessor for an experiment
# # Try 'paa', 'fft', or 'cwt'
# preprocessor = TTBIPreprocessor(method='cwt', n_segments=512, cwt_scales=64)

# # 3. Transform and Fit the scaler (Use fit_scaler=True for your Training fold!)
# X_processed = preprocessor.transform(X_raw, fit_scaler=True)

# # For CWT, X_processed is now shape (Samples, 2 Channels, 64 Scales, 512 Length)
# # For PAA, X_processed is now shape (Samples, 2 Channels, 512 Length)

# # 4. Save scaler so the online DT can load it later
# preprocessor.save_scaler('cwt_scaler.pkl')

In [ ]:
# --- Hardware Setup ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# 1. The Core Training & Evaluation Function
# =====================================================================
def train_and_evaluate(trial, config, dataset_name, n_epochs):
    """
    Loads data, builds a dynamically sized model, trains it, and returns validation accuracy.
    Includes Optuna pruning to kill unpromising trials early.
    """
    # 1. Load Data (Now dynamically selecting 'bogie' or 'wheel')
    X_raw, y = load_ttbi_dataset(
        dataset_name=dataset_name, 
        sensor_types=config['sensors'],
        dofs=config['dofs'], 
        n_passages=200 
    )
    
    # 2. Preprocess & Scale
    preprocessor = TTBIPreprocessor(method=config['method'], n_segments=512)
    X_processed = preprocessor.transform(X_raw, fit_scaler=True)
    
    # Train/Test Split & DataLoaders...
    # (Keep your existing DataLoader code here)
    X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)
    
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train)), 
                              batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val)), 
                            batch_size=32, shuffle=False)

    # 3. Suggest Hyperparameters...
    # (Keep your existing Optuna hyperparameter suggestion loop here)
    params = {
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        'n_conv_layers': trial.suggest_int('n_conv_layers', 2, 4),
        'n_dense_layers': trial.suggest_int('n_dense_layers', 1, 3),
    }
    
    for i in range(params['n_conv_layers']):
        params[f'n_filters_l{i}'] = trial.suggest_int(f'n_filters_l{i}', 16, 128, step=16)
        params[f'kernel_size_l{i}'] = trial.suggest_categorical(f'kernel_size_l{i}', [3, 5, 7])
        params[f'pooling_l{i}'] = trial.suggest_categorical(f'pooling_l{i}', [True, False])
        
    for i in range(params['n_dense_layers']):
        params[f'n_dense_units_l{i}'] = trial.suggest_int(f'n_dense_units_l{i}', 32, 256, step=32)
        params[f'dropout_l{i}'] = trial.suggest_float(f'dropout_l{i}', 0.1, 0.5)

    # 4. Model Routing! (1D vs 2D)
    in_channels = X_train.shape[1]
    
    if config['model'] == '1D':
        # Dynamically grab the actual sequence length (5831 for raw, 512 for PAA)
        seq_length = X_train.shape[2] 
        
        model = SpaceAware1DCNN(
            n_segments=seq_length,  # <-- Pass the dynamic length here!
            n_classes=61, 
            in_channels=in_channels, 
            params=params, 
            use_space2vec=config['use_space2vec']
        ).to(DEVICE)
        
    elif config['model'] == '2D':
        # CWT data is 4D: (Batch, Channels, Scales, Sequence_Width)
        scales_height = X_train.shape[2]
        seq_width = X_train.shape[3]
        
        model = Simple2DCNN(
            in_channels=in_channels,
            n_classes=61,
            params=params,
            image_height=scales_height, # <-- Pass the dynamic height
            image_width=seq_width       # <-- Pass the dynamic width
        ).to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

    # 5. Training Loop with Pruning
    # print('Starting model training...')
    
    best_val_acc = 0.0
    patience = 5  # How many epochs to wait before stopping
    patience_counter = 0
    
    for epoch in range(n_epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                outputs = model(batch_X)
                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
                
        val_acc = correct / total
                
        # 1. Optuna Pruning (Kills trials that are worse than the median)
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        # 2. Standard Early Stopping (Kills trials that have plateaued)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0  # Reset counter
        else:
            patience_counter += 1
            if patience_counter >= patience:
                # print(f"Trial plateaued at epoch {epoch}. Stopping early.")
                break # Exit the epoch loop early!

    return best_val_acc

# =====================================================================
# 2. Objective Wrapper for Optuna
# =====================================================================
class Objective:
    def __init__(self, config, dataset_name, n_epochs):
        self.config = config
        self.dataset_name = dataset_name
        self.n_epochs = n_epochs

    def __call__(self, trial):
        return train_and_evaluate(trial, self.config, self.dataset_name, self.n_epochs)

# =====================================================================
# 3. Main Tournament Execution
# =====================================================================
if __name__ == "__main__":
    
    # --- The Ultimate Ablation Matrix ---
    ablation_grid = [
        # --- 1. BASELINES (Sprung Mass / Bogie) ---
        {'name': 'Raw_1DOF_Bogie', 'method': 'raw', 'sensors': ['bogie'], 'dofs': [1], 'use_space2vec': False, 'model': '1D'},
        {'name': 'PAA_1DOF_Bogie', 'method': 'paa', 'sensors': ['bogie'], 'dofs': [1], 'use_space2vec': False, 'model': '1D'},
        
        # --- 2. SENSOR TOPOLOGY TESTS ---
        # Unsprung Mass (Wheel Axle)
        {'name': 'PAA_1DOF_Wheel', 'method': 'paa', 'sensors': ['wheel'], 'dofs': [1], 'use_space2vec': False, 'model': '1D'},
        
        # Multi-DOF Sprung (Vertical [1] + Pitch [2])
        {'name': 'PAA_MultiDOF0_Bogie', 'method': 'paa', 'sensors': ['bogie'], 'dofs': [0, 1], 'use_space2vec': False, 'model': '1D'},
        {'name': 'PAA_MultiDOF2_Bogie', 'method': 'paa', 'sensors': ['bogie'], 'dofs': [1, 2], 'use_space2vec': False, 'model': '1D'},
        {'name': 'PAA_AllDOFs_Bogie', 'method': 'paa', 'sensors': ['bogie'], 'dofs': [0, 1, 2], 'use_space2vec': False, 'model': '1D'},
        
        # Multi-DOF Unsprung (Front Axle Vertical [0] + Rear Axle Vertical [1])
        {'name': 'PAA_MultiDOF0_Wheel', 'method': 'paa', 'sensors': ['wheel'], 'dofs': [0, 1], 'use_space2vec': False, 'model': '1D'},
        {'name': 'PAA_MultiDOF2_Wheel', 'method': 'paa', 'sensors': ['wheel'], 'dofs': [1, 2], 'use_space2vec': False, 'model': '1D'},
        {'name': 'PAA_AllDOFs_Wheel', 'method': 'paa', 'sensors': ['wheel'], 'dofs': [1, 2], 'use_space2vec': False, 'model': '1D'},

        # --- 3. SENSOR FUSION (Stacking Bogie + Wheel) ---
        # Vertical only (2 Channels: Bogie Vert, Wheel Vert)
        {'name': 'PAA_Fusion_1DOF', 'method': 'paa', 'sensors': ['bogie', 'wheel'], 'dofs': [1], 'use_space2vec': False, 'model': '1D'},
        # Vertical + Pitch (4 Channels: Bogie Vert, Bogie Pitch, Wheel Vert, Wheel Pitch)
        {'name': 'PAA_Fusion_MultiDOF', 'method': 'paa', 'sensors': ['bogie', 'wheel'], 'dofs': [1, 2], 'use_space2vec': False, 'model': '1D'},
        {'name': 'PAA_Fusion_AllDOFs', 'method': 'paa', 'sensors': ['bogie', 'wheel'], 'dofs': [0, 1, 2], 'use_space2vec': False, 'model': '1D'},

        # --- 4. SPATIAL CONTEXT (Space2Vec) ---
        {'name': 'PAA_1DOF_Bogie_S2V', 'method': 'paa', 'sensors': ['bogie'], 'dofs': [1], 'use_space2vec': True, 'model': '1D'},
        {'name': 'PAA_Fusion_MultiDOF_S2V', 'method': 'paa', 'sensors': ['bogie', 'wheel'], 'dofs': [1, 2], 'use_space2vec': True, 'model': '1D'},

        # --- 5. ADVANCED TIME-FREQUENCY (CWT + 2D CNN) ---
        {'name': 'CWT_1DOF_Bogie', 'method': 'cwt', 'sensors': ['bogie'], 'dofs': [1], 'use_space2vec': False, 'model': '2D'},
        {'name': 'CWT_Fusion_1DOF', 'method': 'cwt', 'sensors': ['bogie', 'wheel'], 'dofs': [1], 'use_space2vec': False, 'model': '2D'},
        {'name': 'CWT_Fusion_AllDOFs', 'method': 'cwt', 'sensors': ['bogie', 'wheel'], 'dofs': [0, 1, 2], 'use_space2vec': False, 'model': '2D'},
    ]

    # --- Phase 1: The Heats (Micro-BO) ---
    print("\n" + "="*50)
    print("PHASE 1: THE HEATS (Micro Bayesian Optimization)")
    print("="*50)
    
    # We use a moderately complex dataset to weed out weak architectures
    heat_dataset = 'data_noise_vehicle_temperature'
    micro_trials = 20
    micro_epochs = 15
    
    results = []

    for config in ablation_grid:
        print(f"\nEvaluating Architecture: {config['name']}")
        
        # MedianPruner kills runs that fall below the median of previous trials
        pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
        study = optuna.create_study(direction='maximize', pruner=pruner)
        
        objective = Objective(config=config, dataset_name=heat_dataset, n_epochs=micro_epochs)
        study.optimize(objective, n_trials=micro_trials)
        
        # Record results
        results.append({
            'name': config['name'],
            'config': config,
            'best_val_acc': study.best_value,
            'best_params': study.best_params
        })
        print(f"  Best Val Accuracy: {study.best_value:.4f}")
        
        study_df = study.trials_dataframe()
        study_df.to_csv(f"optuna_heat_{config['name']}.csv", index=False)

    # --- Ranking ---
    results_df = pd.DataFrame(results).sort_values(by='best_val_acc', ascending=False)
    print("\n--- TOURNAMENT RESULTS ---")
    print(results_df[['name', 'best_val_acc']])

    # --- Phase 2: The Finals (Macro-BO) ---
    print("\n" + "="*50)
    print("PHASE 2: THE FINALS (Heavy Bayesian Optimization)")
    print("="*50)
    
    # We take the top top_n architectures and battle them on the hardest dataset
    top_n = 5
    finals_dataset = 'data_all_variabilities'
    macro_trials = 50
    macro_epochs = 40
    
    for i in range(top_n):
        winner = results_df.iloc[i]
        config = winner['config']
        
        print(f"\nFinalist {i+1}: {config['name']}")
        print(f"Tuning on heavily augmented dataset: {finals_dataset}")
        
        study = optuna.create_study(direction='maximize')
        objective = Objective(config=config, dataset_name=finals_dataset, n_epochs=macro_epochs)
        study.optimize(objective, n_trials=macro_trials)
        
        print(f"\n>>> FINAL CHAMPION RESULTS FOR {config['name']} <<<")
        print(f"  Accuracy on Full Dataset: {study.best_value:.4f}")
        print(f"  Optimal Hyperparameters: {study.best_params}")
        
        # Save the study for this finalist so you can extract params later
        study_df = study.trials_dataframe()
        study_df.to_csv(f"optuna_finals_{config['name']}.csv", index=False)

In [ ]:
# --- ASSUME THESE COME FROM YOUR OPTUNA RESULTS ---
# Replace these with the actual outputs from your winning Optuna trial
champion_config = {'name': 'PAA_MultiDOF_Bogie_S2V', 'method': 'paa', 'sensor': 'bogie', 'dofs': [0, 2], 'use_space2vec': True, 'model': '1D'}
champion_params = {'lr': 0.001, 'weight_decay': 1e-4, 'n_conv_layers': 3, 'n_filters_l0': 64, 'kernel_size_l0': 7, 'pooling_l0': True, 'n_filters_l1': 128, 'kernel_size_l1': 5, 'pooling_l1': True, 'n_filters_l2': 128, 'kernel_size_l2': 3, 'pooling_l2': False, 'n_dense_layers': 2, 'n_dense_units_l0': 128, 'dropout_l0': 0.3, 'n_dense_units_l1': 64, 'dropout_l1': 0.2}

dataset_to_use = 'data_all_variabilities'
epochs = 50
batch_size = 32

print(f"=== TRAINING CHAMPION MODEL: {champion_config['name']} ===")

# 1. Load Data
X_raw, y = load_ttbi_dataset(
    dataset_name=dataset_to_use, 
    sensor_types=champion_config['sensor'], 
    dofs=champion_config['dofs'], 
    n_passages=200 
)

# 2. Train / Val / Test Split (70% / 15% / 15%)
X_temp, X_test, y_temp, y_test = train_test_split(X_raw, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42) # 0.1765 of 0.85 is ~0.15

# 3. Preprocess & Scale (Fit ONLY on Train!)
preprocessor = TTBIPreprocessor(method=champion_config['method'], n_segments=512)
X_train_proc = preprocessor.transform(X_train, fit_scaler=True)
X_val_proc = preprocessor.transform(X_val, fit_scaler=False)
X_test_proc = preprocessor.transform(X_test, fit_scaler=False)

# Save the scaler immediately
preprocessor.save_scaler('scaler.pkl')

# 4. DataLoaders
train_loader = DataLoader(TensorDataset(torch.tensor(X_train_proc).float(), torch.tensor(y_train)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val_proc).float(), torch.tensor(y_val)), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test_proc).float(), torch.tensor(y_test)), batch_size=batch_size, shuffle=False)

# 5. Initialize Model
in_channels = X_train_proc.shape[1]
seq_length = X_train_proc.shape[2]

model = SpaceAware1DCNN(
    n_segments=seq_length, 
    n_classes=61, 
    in_channels=in_channels, 
    params=champion_params, 
    use_space2vec=champion_config['use_space2vec']
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
# Added weight_decay (L2 regularization) to the optimizer
optimizer = optim.Adam(model.parameters(), lr=champion_params['lr'], weight_decay=champion_params.get('weight_decay', 1e-4))

# 6. Training Loop with Model Checkpointing
best_val_loss = float('inf')
patience = 7
patience_counter = 0

for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
        
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            val_loss += criterion(model(batch_X), batch_y).item()
            
    val_loss /= len(val_loader)
    print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f}")
    
    # Save the model if it's the best one so far
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_cnn_model.pth')
        patience_counter = 0 # Reset
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}.")
            break

print("\n--- Training Complete. Evaluating on unseen Test Set ---")

# 7. Generate Predictions on Test Set
# Load the best weights back into the model before evaluating the Test Set!
model.load_state_dict(torch.load('best_cnn_model.pth'))
model.eval()

all_preds = []
all_trues = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(DEVICE)
        outputs = model(batch_X)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_trues.extend(batch_y.numpy())

# 8. Compute Confusion Matrix and CPT
cm = confusion_matrix(all_trues, all_preds, labels=range(61))

# Normalize the confusion matrix over the true labels (rows) to get probabilities
# Add a tiny epsilon to prevent division by zero in case a class has no instances in the test set
cpt = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)

# Save the CPT for the Digital Twin
np.save('conf_mat_dbn.npy', cpt)

# Save the parameters dictionary so the DT can reconstruct the architecture
with open('best_model_params.json', 'w') as f:
    json.dump(champion_params, f)

print("=== ARTIFACTS SAVED FOR DIGITAL TWIN ===")
print(" 1. best_cnn_model.pth (Network Weights)")
print(" 2. scaler.pkl (MinMaxScaler)")
print(" 3. conf_mat_dbn.npy (Conditional Probability Table)")
print(" 4. best_model_params.json (Architecture Hyperparameters)")
print(f"Test Set Accuracy: {np.trace(cm) / np.sum(cm):.4f}")